# **M-Pesa Transaction Analysis**


# Author: Julius Musau


### Project Overview

This project presents an exploratory analysis of my personal M-Pesa transaction history. The aim was to understand my spending habits, identify income sources, discover transaction patterns, and uncover insights that are not readily available from the standard M-Pesa statement.
Through this analysis, I answer questions such as:

- Where does most of my money go?
- Which transaction types dominate my spending?
- Who receives most of my payments?
- When am I most active on M-Pesa?
- Which months have the highest expenditure?
- How much do I spend on transaction charges?
- What trends exist in my financial behaviour?

The project was implemented entirely in Python using Jupyter Notebook.

Dataset Description

The dataset consists of exported M-Pesa transaction statements covering multiple years of personal financial activity.

Each record represents one completed M-Pesa transaction.

### Main Variables

| Variable | Description |
|-----------|-------------|
| Completion Time | Date and time of the transaction |
| Details | Original M-Pesa transaction description |
| Transaction Status | Status of the transaction |
| Paid In | Amount received |
| Withdrawn | Amount spent |
| Balance | Account balance after the transaction |
| Transaction_Type | Simplified transaction category |

The dataset contains different transaction categories including:

- Customer Transfers
- Merchant Payments
- Pay Bills
- Business Payments
- Loan Repayments
- Fuliza Transactions
- Airtime Purchases
- Funds Received
- Overdraft Transactions

# Importing Required Libraries

libraries used throughout the project:

- Pandas
- NumPy
- Matplotlib
- Plotly
- Regular Expressions (re)



In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import re


In [ ]:
df=pd.read_csv('/content/mpesa.csv')
df.head()

,,Completion Time,Transaction Status,Paid In,Withdrawn,Balance,Transaction_Type,Unnamed: 7,Details
0,UEUCU5O5BW,5/30/2026 9:19,Completed,0.0,-10.0,0.0,Pay Bill,NaN,Pay Bill Online Fuliza M-Pesa to 880100 - NCBA...
1,UEUCU5O5BW,5/30/2026 9:19,Completed,10.0,0.0,0.0,OverDraft,NaN,OverDraft of Credit Party
2,UETCU5N8NA,5/29/2026 21:17,Completed,0.0,-23.0,0.0,Pay Bill,NaN,Pay Bill Online Fuliza M-Pesa to 4187665 - DIR...
3,UETCU5N8NA,5/29/2026 21:17,Completed,23.0,0.0,23.0,OverDraft,NaN,OverDraft of Credit Party
4,UETCU5M9ZU,5/29/2026 19:04,Completed,40.0,0.0,40.0,OverDraft,NaN,OverDraft of Credit Party


# Data Cleaning and Preprocessing


The following cleaning steps were undertaken:


- Converted transaction dates into datetime format.
- Extracted separate Date and Time fields.
- Removed commas from monetary values.
- Converted withdrawn amounts into absolute values.
- Checked for missing values.
- Standardized transaction descriptions.
- Converted numeric columns into appropriate data types.

These steps ensured consistency and improved the quality of subsequent analyses.

In [ ]:
# Print the list of column names to see if everything is okay with the columns
df.columns.tolist()

['                                                                                                                                                                     ',
 'Completion Time',
 'Transaction Status',
 'Paid In',
 'Withdrawn',
 'Balance',
 'Transaction_Type',
 'Unnamed: 7',
 'Details']

In [ ]:
# Remove unnamed/blank columns
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Remove columns whose names are only spaces
df = df.loc[:, df.columns.str.strip() != '']


In [ ]:
# Now lets check the data types of the data
df.dtypes

,0
Completion Time,object
Transaction Status,object
Paid In,float64
Withdrawn,float64
Balance,float64
Transaction_Type,object
Details,object


In [ ]:
# Convert Paid In

df['Paid In'] = pd.to_numeric(
    df['Paid In'],
    errors='coerce'
)

In [ ]:
# Convert Withdrawn to numeric

df['Withdrawn'] = (
    df['Withdrawn']
      .astype(str)
      .str.replace(',', '')
      .astype(float)
      .abs()
)

In [ ]:

# Convert datetime

df['Completion_Time'] = pd.to_datetime(
    df['Completion Time'],
    errors='coerce'
)

# Converting the 'Completion_Time' to datetime format and split into 'Date' and 'Time' so that I can work with it better

In [ ]:
# Date Features

df['Date'] = df['Completion_Time'].dt.date
df['Time'] = df['Completion_Time'].dt.time
df['Year'] = df['Completion_Time'].dt.year
df['Month Name'] = df['Completion_Time'].dt.month_name()
df['Year-Month'] = df['Completion_Time'].dt.strftime('%B %Y')

# Transaction Hour

df['Hour'] = df['Completion_Time'].dt.hour

In [ ]:

# TIME PERIODS
def time_period(hour):

    if 0 <= hour < 6:
        return "Midnight"

    elif 6 <= hour < 12:
        return "Morning"

    elif 12 <= hour < 18:
        return "Afternoon"

    else:
        return "Evening"

df['Time_Period'] = df['Hour'].apply(time_period)

#  Exploratory Data Analysis (EDA)

Exploratory Data Analysis (EDA) was conducted to better understand my transaction history, identify spending patterns, examine income sources, and discover behavioural trends hidden within the M-Pesa statement.




# Overall Cash Flow Analysis
### Metrics Calculated

- Total Money Received
- Total Money Spent
- Net Cash Flow

In [ ]:
total_recieved=df['Paid In'].sum()
print(f"Total Money Received KES  {total_recieved:,.2f} .")

Total Money Received KES  841,413.98 .


In [ ]:
total_spent=df['Withdrawn'].sum()
print(f"Total Money Spent KES  {total_spent:,.2f} .")

Total Money Spent KES  1,349,341.94 .


In [ ]:
total_earnings=total_recieved-total_spent
print(f"Net Cash Flow {total_earnings:,.2f} .")

Net Cash Flow -507,927.96 .


In [ ]:
# At first glance this looks like I spent more money than I received 😂.

# So basically I finished the year with a deficit of KSHS 507,927.96

# However, this does not necessarily mean I lived beyond my means.

# Some of my transactions were financed through linked bank accounts, Fuliza overdrafts, and other banking services connected to my M-Pesa account.

# Therefore, not all spending was funded by money directly received through M-Pesa.

# 8.2 Transaction Category Distribution


M-Pesa supports several transaction services such as Customer Transfers, Merchant Payments, Business Payments, Pay Bills, Loan Repayments, and Fuliza transactions.

This analysis examines the frequency of each transaction category to understand which services were used most frequently.

Understanding transaction frequency provides insight into everyday financial behaviour.

In [ ]:

# TRANSACTION TYPE ANALYSIS

transaction_totals = (
    df.groupby('Transaction_Type')['Withdrawn']
      .sum()
      .sort_values(ascending=False)
)

transaction_percentages = (
    transaction_totals /
    transaction_totals.sum()
) * 100

transaction_summary = pd.DataFrame({
    'Amount Spent (KES)': transaction_totals,
    'Percentage (%)': transaction_percentages
})

transaction_summary

,Amount Spent (KES),Percentage (%)
Transaction_Type,,
Customer Transfer,504134.68,37.365121
Pay Bill,240750.36,17.843776
Other,238210.29,17.655513
Merchant Payment,181987.28,13.488413
Funds received,65996.30,4.891470
Loan Repayment,56648.25,4.198618
Pay Bill Charge,20734.86,1.536813
Customer Transfer Charge,17053.96,1.263994
Bundle Purchase,16017.22,1.187154


In [ ]:
# The largest share of my spending went to Customer Transfer, where I spent KES 504,134.68.

# The smallest amount was spent on OverDraft, totalling only KES 3,688.00.

# Looking at the distribution, Customer Transfers and Pay Bills account for the majority of my M-Pesa expenditure.

# This suggests that most of my transactions involve sending money to individuals and making bill payments rather than merchant purchases.

# On the bright side, at least I now know exactly where my money disappears every month!

# Let's visualize the spending distribution for a clearer understanding.

In [ ]:

# SPENDING VISUALIZATION


plot_df = transaction_summary.reset_index()

plot_df['Percentage_Text'] = (
    plot_df['Percentage (%)']
    .apply(lambda x: f"{x:.1f}%")
)

fig = px.bar(
    plot_df,
    x='Transaction_Type',
    y='Amount Spent (KES)',
    color='Transaction_Type',
    text='Percentage_Text',
    title='Total Amount Spent by Transaction Type',
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig.update_layout(
    title_x=0.5,
    template='plotly_white',
    yaxis_tickformat=',.0f'
)

fig.show()

In [ ]:
# One thing that immediately stood out was how much of my money went towards Customer Transfers.

# It seems sending money to friends and family was one of my biggest financial habits 😂.
# We will see which person I sent the most amount of money later on 😊😊😊

# Pay Bills came in second, which makes sense since most of my bank transfers,

# loan repayments, and other recurring payments were processed through Paybill numbers.


# Merchant Payments also accounted for a significant portion of my expenditure,
# showing that I frequently used M-Pesa instead of cash when making purchases.

# Interestingly, Loan Repayments and OverDraft transactions also contributed to my spending,
# reminding me how often I relied on Fuliza before paying it back.

# I don't know why I spent so little on Airtime. Perhaps its because I have very few friends😂😂😂

# Anyway, let's go ahead and look deeper into my monthly spending patterns

# Monthly Spending Analysis
Monthly spending patterns help identify periods of increased or reduced expenditure.

Grouping transactions by month allows seasonal trends and changes in spending behaviour to be observed over time.


In [ ]:

# MONTHLY SPENDING

monthly_spending = (
    df.groupby('Year-Month')['Withdrawn']
      .sum()
      .reset_index()
)
monthly_spending

,Year-Month,Withdrawn
0,April 2025,24773.00
1,April 2026,41504.53
2,August 2025,28394.00
3,December 2024,33975.00
4,December 2025,42525.14
5,February 2025,252070.32
6,February 2026,34594.55
7,January 2025,201123.84
8,January 2026,27512.72
9,July 2025,32241.00


In [ ]:
# The monthly spending analysis reveals several interesting spending patterns.

# September 2024 recorded my lowest monthly expenditure at KES 14,459.00,
# suggesting that this was my most financially conservative month.

# Spending then increased gradually before reaching exceptionally high levels
# at the beginning of 2025. January 2025 recorded KES 201,123.84 while
# February 2025 became the highest spending month in the entire dataset at
# KES 252,070.32. These two months stand out as clear outliers compared to
# the rest of the analysis period.

# After February 2025, my spending returned to more typical monthly levels,
# fluctuating between approximately KES 20,000 and KES 46,000.

# Another interesting observation appears in 2026. From January through May,
# my monthly expenditure increased consistently every month—from
# KES 27,512.72 in January to KES 45,343.48 in May.

# This steady upward trend may indicate increasing financial   changes in my
# spending habits over time.


In [ ]:
fig = px.bar(
    monthly_spending,
    x='Year-Month',
    y='Withdrawn',
    color='Year-Month',
    title='Monthly Spending Trend',
    color_discrete_sequence=px.colors.sequential.Turbo
)

fig.update_layout(
    title_x=0.5,
    yaxis_tickformat=',.0f'
)

fig.show()

# Highest Spending Day


Daily expenditure analysis identifies the single day during which the highest amount of money was spent.

This provides insight into exceptional spending events and highlights days with unusually high financial activity.

The transactions that contributed to the highest daily expenditure are also displayed.

In [ ]:
# Analysing my daily transactions on M-Pesa the day i spent the most

# HIGHEST SPENDING DAY

daily_spending = (
    df.groupby('Date')['Withdrawn']
      .sum()
      .reset_index()
)

highest_day = daily_spending.loc[
    daily_spending['Withdrawn'].idxmax()
]


print("HIGHEST SPENDING DAY")


print(
    f" Date: {highest_day['Date']}"
)

print(
    f" Amount: KES {highest_day['Withdrawn']:,.2f}"
)

HIGHEST SPENDING DAY
 Date: 2025-02-09
 Amount: KES 51,020.38


# 11. Top Recipients Analysis


Understanding where money is spent is just as important as understanding how much is spent.
Recipient names were extracted from the original M-Pesa transaction descriptions using pattern matching techniques.The cumulative amount sent to each recipient was then calculated to identify the individuals and businesses that received the largest share of outgoing payments.

The Top 10 recipients are presented below.

In [ ]:

# RECIPIENT EXTRACTION


def extract_recipient(detail):

    detail = str(detail)

    patterns = [

        r'Customer Transfer(?: Fuliza MPesa)? to - .*? ([A-Za-z\s]+)$',

        r'Merchant Payment(?: Fuliza M-Pesa(?: Online)?)? to \d+ -\s*(.*)',

        r'Pay Bill(?: Online)?(?: Fuliza M-Pesa)? to \d+ -\s*(.*)',

        r'Customer Payment to Small Business to - .*? ([A-Za-z\s]+)$'
    ]

    for pattern in patterns:

        match = re.search(pattern, detail)

        if match:

            return match.group(1).strip()

    return None


df['Recipient'] = df['Details'].apply(
    extract_recipient
)

In [ ]:

# TOP RECIPIENTS


df_spent = df[
    (df['Withdrawn'] > 0)
    &
    (df['Recipient'].notna())
]

top10 = (

    df_spent.groupby('Recipient')['Withdrawn']

    .agg(['sum', 'count'])

    .reset_index()

    .sort_values('sum', ascending=False)

    .head(10)

)

top10 = (
    df[df['Recipient'].notna()]
    .groupby('Recipient')['Withdrawn']
    .agg(['sum', 'count'])
    .sort_values('sum', ascending=False)
    .head(10)
)

top10

,sum,count
Recipient,,
ali abdul,127099.96,671
MATHEW MUSYOKA KITHUMBI,41409.58,21
Co-operative Bank Money Transfer Acc.,32415.00,14
KCB Paybill AC Acc. 1110571976,29250.00,7
alice kimeu,23508.02,108
Ngooni Supermarkets .1,22451.00,35
Johnstone Kimilu,21386.48,74
MARIAN WAEMA,14480.00,18
BETH MWEU,13475.58,13


# 12. Time-of-Day Transaction Analysis

## Objective

Financial behaviour often varies throughout the day.

To investigate transaction timing, every transaction was classified into one of four time periods:

| Time Period | Time Range |
|--------------|------------|
| Midnight | 00:00 – 05:59 |
| Morning | 06:00 – 11:59 |
| Afternoon | 12:00 – 17:59 |
| Evening | 18:00 – 23:59 |

The number of transactions occurring within each period was calculated to determine the most active time of day.

In [ ]:

# TIME OF DAY ANALYSIS
time_summary = (
    df.groupby('Time_Period')
      .size()
      .sort_values(ascending=False)
)

total_transactions = len(df)


print("TRANSACTION ACTIVITY BY TIME OF DAY")


print(time_summary)

print("\n")

print(
    f"📊 Total Transactions: "
    f"{total_transactions:,}"
)

most_active = time_summary.idxmax()

print(
    f"⏰ Most Active Period: "
    f"{most_active}"
)

TRANSACTION ACTIVITY BY TIME OF DAY
Time_Period
Evening      4477
Afternoon    1735
Midnight     1134
Morning       944
dtype: int64


📊 Total Transactions: 8,290
⏰ Most Active Period: Evening


In [ ]:
# Looking at the results, one pattern is immediately noticeable.
#
# Most of my financial activity happens during the evening hours.
#
# Out of 8,290 recorded transactions, 4,477 were made between
# 6:00 PM and 11:59 PM, accounting for more than half of all
# the transactions in the dataset.
#
# This is not entirely surprising since evenings are usually
# when most people have finished work or school and have time
# to shop, settle bills, transfer money, or make other personal
# payments.
# Interestingly, the relatively high number of midnight
# transactions suggests that I occasionally make payments late
# at night, perhaps for online services, emergency transfers,
# or because M-Pesa is available 24/7.
#
# Overall, this analysis shows that my financial activity is
# heavily concentrated in the second half of the day, with the
# evening being by far my busiest transaction period.
#


Future Improvements

Potential enhancements for this project include:

- Building an interactive Streamlit dashboard.
- Developing a Power BI version of the dashboard.
- Integrating predictive models to forecast future expenditure.
- Applying machine learning techniques for automatic expense categorization.
- Creating a personal finance recommendation system based on historical spending behaviour.